In [1]:
import pandas as pd
import numpy as np

# Load cleaned data
df = pd.read_csv("clean_data_after_eda.csv")

# =========================
# 1. REMOVE USELESS COLUMNS
# =========================
# Drop columns with only 1 unique value
for col in df.columns:
    if df[col].nunique() <= 1:
        df.drop(columns=col, inplace=True)

# Drop ID column (not useful for ML)
if 'id' in df.columns:
    df.drop(columns='id', inplace=True)

# =========================
# 2. DATE FEATURE EXTRACTION
# =========================
date_cols = ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        df[col + '_year'] = df[col].dt.year
        df[col + '_month'] = df[col].dt.month
        df[col + '_day'] = df[col].dt.day

# =========================
# 3. CREATE NEW FEATURES
# =========================

# Contract duration (in days)
if 'date_activ' in df.columns and 'date_end' in df.columns:
    df['contract_duration'] = (df['date_end'] - df['date_activ']).dt.days

# Time since last modification
if 'date_modif_prod' in df.columns and 'date_renewal' in df.columns:
    df['time_since_modif'] = (df['date_renewal'] - df['date_modif_prod']).dt.days

# Consumption ratio
if 'cons_12m' in df.columns and 'cons_last_month' in df.columns:
    df['consumption_ratio'] = df['cons_last_month'] / (df['cons_12m'] + 1)

# Forecast difference
if 'forecast_cons_12m' in df.columns and 'cons_12m' in df.columns:
    df['forecast_diff'] = df['forecast_cons_12m'] - df['cons_12m']

# Price difference
if 'forecast_price_energy_peak' in df.columns and 'forecast_price_energy_off_peak' in df.columns:
    df['price_diff'] = df['forecast_price_energy_peak'] - df['forecast_price_energy_off_peak']

# Net margin ratio
if 'net_margin' in df.columns and 'cons_12m' in df.columns:
    df['margin_per_consumption'] = df['net_margin'] / (df['cons_12m'] + 1)

# =========================
# 4. DROP ORIGINAL DATE COLUMNS
# =========================
df.drop(columns=[col for col in date_cols if col in df.columns], inplace=True)

# =========================
# 5. HANDLE CATEGORICAL DATA
# =========================
df = pd.get_dummies(df, drop_first=True)

# =========================
# 6. FINAL CHECK
# =========================
print("Final Shape:", df.shape)
print(df.head())

# =========================
# 7. SAVE FINAL DATA
# =========================
df.to_csv("final_feature_engineered_data.csv", index=False)

Final Shape: (14606, 67)
   cons_12m  cons_gas_12m  cons_last_month  forecast_cons_12m  \
0         0         54946                0               0.00   
1      4660             0                0             189.95   
2       544             0                0              47.96   
3      1584             0                0             240.04   
4      4425             0              526             445.75   

   forecast_cons_year  forecast_discount_energy  forecast_meter_rent_12m  \
0                   0                       0.0                     1.78   
1                   0                       0.0                    16.27   
2                   0                       0.0                    38.72   
3                   0                       0.0                    19.83   
4                 526                       0.0                   131.73   

   forecast_price_energy_off_peak  forecast_price_energy_peak  \
0                        0.114481                    0.098142 